# QC A3 Parity Pilot

This notebook validates the Investintell A3 path inside QuantConnect Research without using QC FRED or returns as an A3 objective. It consumes immutable Investintell L2/revision-uncertainty/config artifacts and compares normalized outputs against the local worker bundle.

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import platform
import subprocess
import sys
import time
import tracemalloc

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from qc_a3_core import A3ParityConfig, materialize_harness_source_from_manifest, read_json, run_parity, write_json


def ensure_git_repo_for_qc_research(project_root: Path) -> bool:
    probe = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        cwd=project_root,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    if probe.returncode == 0:
        return False
    init = subprocess.run(
        ["git", "init"],
        cwd=project_root,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    if init.returncode != 0:
        raise RuntimeError(f"Unable to initialize git repository for QC Research: {init.stderr}")
    return True


GIT_INITIALIZED_FOR_QC_RESEARCH = ensure_git_repo_for_qc_research(PROJECT_ROOT)


## Object Store contract

Upload every file listed in `object_store_manifest.json` to its matching Object Store key. The core A3 run must read only these Investintell artifacts. QC History/FRED is intentionally absent from this notebook.

In [ ]:
DEFAULT_OBJECT_STORE_MANIFEST_KEY = "investintell/a3/qc-a3-parity/1138754/d9f029f7a4d450ea396504ad/object_store_manifest.json"
OBJECT_STORE_MANIFEST_KEY_FILE = Path("object_store_manifest_key.txt")
OBJECT_STORE_MANIFEST_KEY = os.environ.get("QC_A3_OBJECT_STORE_MANIFEST_KEY")
if not OBJECT_STORE_MANIFEST_KEY and OBJECT_STORE_MANIFEST_KEY_FILE.exists():
    OBJECT_STORE_MANIFEST_KEY = OBJECT_STORE_MANIFEST_KEY_FILE.read_text(encoding="utf-8").strip()
if not OBJECT_STORE_MANIFEST_KEY:
    OBJECT_STORE_MANIFEST_KEY = DEFAULT_OBJECT_STORE_MANIFEST_KEY
REQUIRE_QC_CLOUD = os.environ.get("QC_A3_REQUIRE_QC_CLOUD", "1").strip().lower() not in {"0", "false", "no"}
_RESEARCH_NODE_ENV = os.environ.get("QC_NODE_TYPE") or os.environ.get("QC_NODE_NAME")
RESEARCH_NODE = _RESEARCH_NODE_ENV or "qc_research"
RESEARCH_NODE_SOURCE = "env" if _RESEARCH_NODE_ENV else "fallback"
PLATFORM_TEXT = platform.platform()

def maybe_quantbook_object_store():
    try:
        from QuantConnect import TimeZones
        from QuantConnect.Research import QuantBook
    except Exception:
        return None
    try:
        qb = QuantBook()
        qb.set_start_date(2014, 2, 19)
        qb.set_end_date(2026, 6, 24)
        qb.set_time_zone(TimeZones.NewYork)
    except Exception:
        return None
    return getattr(qb, "object_store", getattr(qb, "ObjectStore", None))

object_store = maybe_quantbook_object_store()

def object_store_file_path(key: str) -> Path | None:
    if object_store is None:
        return None
    if hasattr(object_store, "get_file_path"):
        return Path(object_store.get_file_path(key))
    if hasattr(object_store, "GetFilePath"):
        return Path(object_store.GetFilePath(key))
    raise RuntimeError("QuantConnect Object Store does not expose get_file_path/GetFilePath")

manifest_path = None
bundle_source = None
local_bundle_fallback_used = False
if OBJECT_STORE_MANIFEST_KEY:
    manifest_path = object_store_file_path(OBJECT_STORE_MANIFEST_KEY)
    if manifest_path is not None:
        bundle_source = "quantbook_object_store"
if manifest_path is None:
    local_bundle_fallback_used = True
    bundle_source = "local_bundle_fallback"
    manifest_path = Path("_tmp_qc_a3_parity/object_store_manifest.json")
execution_backend = "quantconnect_cloud_research" if not local_bundle_fallback_used else "local_research_fallback"
if REQUIRE_QC_CLOUD:
    if not OBJECT_STORE_MANIFEST_KEY:
        raise RuntimeError("QC cloud mode requires object_store_manifest_key.txt or QC_A3_OBJECT_STORE_MANIFEST_KEY")
    if local_bundle_fallback_used:
        raise RuntimeError("QC cloud mode requires bundle_source=quantbook_object_store; local fallback was used")
    if RESEARCH_NODE is None:
        raise RuntimeError("QC cloud mode requires research_node from QC_NODE_TYPE/QC_NODE_NAME")
    if "WSL2" in PLATFORM_TEXT:
        raise RuntimeError(f"QC cloud mode cannot run on WSL2 platform: {PLATFORM_TEXT}")

manifest = read_json(manifest_path)
manifest["_local_bundle_dir"] = str(manifest_path.parent)
assert manifest["bridge_scope"] == "qc_research_parity_only"
assert manifest["runtime_activation"] is False
assert manifest["a3_status"] == "open_macro_v03"
assert manifest["upload_policy"]["parquet_upload_allowed"] is False
materialize_harness_source_from_manifest(manifest, object_store_file_path, project_root=PROJECT_ROOT)
manifest["selected"]


In [ ]:
tracemalloc.start()
total_started = time.perf_counter()
load_started = time.perf_counter()

bundle_dir = manifest_path.parent

def artifact_path(name: str) -> Path:
    item = manifest["object_files"][name]
    path = object_store_file_path(item["object_store_key"])
    if path is not None:
        return path
    return bundle_dir / item["relative_path"]

feature_manifest_path = artifact_path("feature_manifest")
revision_uncertainty_manifest_path = artifact_path("revision_uncertainty_manifest")
config_catalog_path = artifact_path("config_catalog_normalized")
macro_l2_npz_path = artifact_path("macro_l2_union_numeric")
revision_uncertainty_npz_path = artifact_path("revision_uncertainty_numeric")
object_store_load_seconds = time.perf_counter() - load_started

results_dir = Path("results")
config = A3ParityConfig(
    feature_manifest=feature_manifest_path,
    revision_uncertainty_manifest=revision_uncertainty_manifest_path,
    config_catalog=config_catalog_path,
    a32_grid_dir=feature_manifest_path.parent,
    output_dir=results_dir,
    expected_v03_grid_dir=bundle_dir,
    macro_l2_npz=macro_l2_npz_path,
    revision_uncertainty_npz=revision_uncertainty_npz_path,
    a31_name=manifest["selected"]["a31_config_name"],
    a32_name=manifest["selected"]["a32_config_name"],
    worker_commit=manifest["worker_commit"],
)
scoring_started = time.perf_counter()
report = run_parity(config)
scoring_seconds = time.perf_counter() - scoring_started
current_bytes, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
total_seconds = time.perf_counter() - total_started
report["external_macro_access"] = False
report["execution_backend"] = execution_backend
report["bundle_source"] = bundle_source
report["local_bundle_fallback_used"] = local_bundle_fallback_used
report["research_node"] = RESEARCH_NODE
report["research_node_source"] = RESEARCH_NODE_SOURCE
report["git_initialized_for_qc_research"] = GIT_INITIALIZED_FOR_QC_RESEARCH
report["object_store_prefix_immutable"] = manifest["object_store_prefix_immutable"]
report["timings"] = {
    "object_store_load_seconds": object_store_load_seconds,
    "scoring_time_seconds": scoring_seconds,
    "metrics_time_seconds": None,
    "total_wall_clock_seconds": total_seconds,
}
report["memory"] = {"peak_tracemalloc_bytes": peak_bytes, "current_tracemalloc_bytes": current_bytes}
write_json(results_dir / "qc_cloud_parity_report.json", report)
report["comparison"]


In [ ]:
assert report["runtime_activation"] is False
assert report["a4_status"] == "harness_ready_provisional_A3"
assert report["a5_status"] == "blocked"
assert report["external_macro_access"] is False
assert report["metric_rows_logical_hash"] == report["metrics_canonical_logical_hash"]
assert report["runtime_row_count"] == manifest["expected"]["runtime_row_count"]
assert report["counterfactual_row_count"] == manifest["expected"]["counterfactual_row_count"]
assert report["metric_row_count"] == manifest["expected"]["metric_row_count"]
assert report["runtime_replay_logical_hash"] == manifest["expected"]["macro_runtime_replay_logical_hash"]
assert report["counterfactual_replay_logical_hash"] == manifest["expected"]["macro_counterfactual_replay_logical_hash"]
assert report["metric_rows_logical_hash"] == manifest["expected"]["macro_metric_rows_logical_hash"]
assert report["metrics_canonical_logical_hash"] == manifest["expected"]["macro_metric_rows_canonical_logical_hash"]
assert report["comparison"].get("mismatch_count", 0) == 0
if REQUIRE_QC_CLOUD:
    assert report["execution_backend"] == "quantconnect_cloud_research"
    assert report["bundle_source"] == "quantbook_object_store"
    assert report["local_bundle_fallback_used"] is False
    assert report["research_node"] is not None
    assert "WSL2" not in PLATFORM_TEXT

def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

environment = {
    "research_node": RESEARCH_NODE,
    "research_node_source": RESEARCH_NODE_SOURCE,
    "git_initialized_for_qc_research": GIT_INITIALIZED_FOR_QC_RESEARCH,
    "lean_version": os.environ.get("LEAN_VERSION"),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "execution_backend": execution_backend,
    "bundle_source": bundle_source,
    "local_bundle_fallback_used": local_bundle_fallback_used,
    "numpy_version": package_version("numpy"),
    "pandas_version": package_version("pandas"),
    "object_store_prefix_immutable": manifest["object_store_prefix_immutable"],
    "timings": report["timings"],
    "memory": report["memory"],
}
write_json(results_dir / "qc_cloud_environment.json", environment)
print(report["runtime_replay_logical_hash"])
print(report["metric_rows_logical_hash"])
